In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install wandb -q

In [ ]:
!pip install lightgbm -q

In [ ]:
import torch
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import wandb

wandb.login()

In [ ]:
train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample_submission=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [ ]:
# FIXED COLUMN HANDLING
QUESTION = "question" if "question" in train.columns else train.columns[0]
A, B, C, D, E = "A", "B", "C", "D", "E"
ANSWER = "answer" if "answer" in train.columns else "label"

In [ ]:
# BUILD DATA (QUESTION-OPTION PAIRS)
def build_data(df):
    X_text = []
    y = []

    for _, r in df.iterrows():
        q = str(r[QUESTION])

        options = {
            "A": r[A],
            "B": r[B],
            "C": r[C],
            "D": r[D],
            "E": r[E],
        }

        for k, v in options.items():
            X_text.append(q + " " + str(v))
            y.append(1 if k == r[ANSWER] else 0)

    return X_text, np.array(y)

In [ ]:
# METRIC
def evaluate(model, X, y):
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds)
    return acc, f1

In [ ]:
# TRAIN EXPERIMENT
def run_experiment(run_name, params, X_train, y_train, X_val, y_val, vectorizer):

    wandb.init(project="mcq-lightgbm", name=run_name, config=params)

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    train_acc, train_f1 = evaluate(model, X_train, y_train)
    val_acc, val_f1 = evaluate(model, X_val, y_val)

    print(f"\n=== {run_name} ===")
    print(f"Train Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
    print(f"Val Acc:   {val_acc:.4f}, F1: {val_f1:.4f}")

    wandb.log({
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_acc": val_acc,
        "val_f1": val_f1
    })

    wandb.finish()

    return model


In [ ]:
!pip install lightgbm -q

In [ ]:
# print(lgb.__version__)

In [ ]:
# SPLIT BY QUESTION (IMPORTANT FIX)
from sklearn.model_selection import train_test_split
questions = np.arange(len(train))

train_idx, val_idx = train_test_split(
    questions,
    test_size=0.2,
    random_state=42
)

train_df = train.iloc[train_idx].reset_index(drop=True)
val_df = train.iloc[val_idx].reset_index(drop=True)

# BUILD DATA ON SPLITS
X_train_text, y_train = build_data(train_df)
X_val_text, y_val = build_data(val_df)

# TF-IDF (SINGLE FIT)
vectorizer = TfidfVectorizer(max_features=100000, ngram_range=(1, 2))

X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)

In [ ]:
params1 = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 64
}

params2 = {
    "n_estimators": 600,
    "learning_rate": 0.03,
    "num_leaves": 128
}

params3 = {
    "n_estimators": 500,
    "learning_rate": 0.04,
    "num_leaves": 96,
    "subsample": 0.8
}

In [ ]:
model1 = run_experiment("run1", params1, X_train, y_train, X_val, y_val, vectorizer)
model2 = run_experiment("run2", params2, X_train, y_train, X_val, y_val, vectorizer)
model3 = run_experiment("run3", params3, X_train, y_train, X_val, y_val, vectorizer)

In [ ]:
# PREDICTION (ENSEMBLE)
def predict(models, vectorizer, df):

    preds = []

    for _, r in df.iterrows():
        q = str(r[QUESTION])

        options = {
            "A": r[A],
            "B": r[B],
            "C": r[C],
            "D": r[D],
            "E": r[E],
        }

        scores = {}

        for k, v in options.items():
            text = q + " " + str(v)
            vec = vectorizer.transform([text])

            prob = np.mean([m.predict_proba(vec)[0][1] for m in models])
            scores[k] = prob

        top3 = sorted(scores, key=scores.get, reverse=True)[:3]
        preds.append(" ".join(top3))

    return preds


In [ ]:
# FINAL SUBMISSION
models = [model1, model2, model3]

preds = predict(models, vectorizer, test)

sub = sample_submission.copy()
sub["Prediction"] = preds

print(sub.head())
sub.to_csv("submission.csv", index=False)

In [ ]:
# import os 
# import re 
# import numpy as np 
# import pandas as pd 
# from collections import Counter 
# import torch 
# import torch.nn as nn 
# from torch.utils.data import Dataset, DataLoader 
# from sklearn.model_selection import train_test_split

**Dataset**

In [ ]:
# train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
# sample_submission=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [ ]:
# train.shape,test.shape

In [ ]:
# train.head(3)

In [ ]:
# test.head(2)

In [ ]:
# train.info()

**Model-1**:
Attention-based BiLSTM Ranker trained from scratch using BCE loss.

**config**

In [ ]:
# MAX_LEN = 256 
# EMBED_DIM = 256 
# HIDDEN_DIM = 256 
# BATCH_SIZE = 64 
# EPOCHS = 10 
# LR = 1e-3 
# LABELS = ["A", "B", "C", "D", "E"]

In [ ]:
# #Tokenizer

# def clean_text(text):
#     text = str(text).lower()
#     text = re.sub(r"[^a-z0-9 ]", " ", text)
#     text = re.sub(r"\s+", " ", text).strip()
#     return text

# def tokenize(text):
#     return clean_text(text).split()


In [ ]:
# #Vocab

# def build_vocab(df):

#     counter = Counter()

#     for _, row in df.iterrows():

#         counter.update(tokenize(row["prompt"]))

#         for col in LABELS:
#             counter.update(tokenize(row[col]))

#     vocab = {
#         "<PAD>": 0,
#         "<UNK>": 1
#     }

#     for word, freq in counter.items():
#         if freq >= 2:
#             vocab[word] = len(vocab)

#     return vocab

# def encode(text, vocab):

#     tokens = tokenize(text)

#     ids = [vocab.get(t, 1) for t in tokens]

#     ids = ids[:MAX_LEN]

#     if len(ids) < MAX_LEN:
#         ids += [0] * (MAX_LEN - len(ids))

#     return ids


In [ ]:
# #Dataset

# class MCQDataset(Dataset):

#     def __init__(self, df, vocab, train_mode=True):

#         self.samples = []

#         for _, row in df.iterrows():

#             prompt = str(row["prompt"])

#             if train_mode:

#                 answer = row["answer"]

#                 for label in LABELS:

#                     option = str(row[label])

#                     text = prompt + " [SEP] " + option

#                     target = 1 if label == answer else 0

#                     self.samples.append(
#                         (
#                             encode(text, vocab),
#                             target
#                         )
#                     )

#             else:

#                 question_id = row["id"]

#                 for label in LABELS:

#                     option = str(row[label])

#                     text = prompt + " [SEP] " + option

#                     self.samples.append(
#                         (
#                             question_id,
#                             label,
#                             encode(text, vocab)
#                         )
#                     )

#         self.train_mode = train_mode

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, idx):

#         if self.train_mode:

#             x, y = self.samples[idx]

#             return (
#                 torch.tensor(x, dtype=torch.long),
#                 torch.tensor(y, dtype=torch.float)
#             )

#         qid, label, x = self.samples[idx]

#         return (
#             qid,
#             label,
#             torch.tensor(x, dtype=torch.long)
#         )

In [ ]:
# # Attention

# class AttentionPool(nn.Module):

#     def __init__(self, hidden_size):

#         super().__init__()

#         self.attn = nn.Linear(hidden_size, 1)

#     def forward(self, x):

#         scores = self.attn(x)

#         weights = torch.softmax(scores, dim=1)

#         pooled = (weights * x).sum(dim=1)

#         return pooled

In [ ]:
# # Model

# class MCQRanker(nn.Module):

#     def __init__(self, vocab_size):

#         super().__init__()

#         self.embedding = nn.Embedding(
#             vocab_size,
#             EMBED_DIM,
#             padding_idx=0
#         )

#         self.lstm = nn.LSTM(
#             EMBED_DIM,
#             HIDDEN_DIM,
#             batch_first=True,
#             bidirectional=True
#         )

#         self.pool = AttentionPool(HIDDEN_DIM * 2)

#         self.classifier = nn.Sequential(
#             nn.Linear(HIDDEN_DIM * 2, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, 1)
#         )

#     def forward(self, x):

#         emb = self.embedding(x)

#         out, _ = self.lstm(emb)

#         pooled = self.pool(out)

#         logits = self.classifier(pooled)

#         return logits.squeeze(-1)

In [ ]:
# #Train...................
# def train_epoch(model, loader, optimizer, criterion):

#     model.train()

#     total_loss = 0

#     for x, y in loader:

#         x = x.to(DEVICE)
#         y = y.to(DEVICE)

#         optimizer.zero_grad()

#         logits = model(x)

#         loss = criterion(logits, y)

#         loss.backward()

#         optimizer.step()

#         total_loss += loss.item()

#     return total_loss / len(loader)

In [ ]:
# #Validation
# @torch.no_grad()
# def evaluate(model, loader, criterion):

#     model.eval()

#     total_loss = 0

#     for x, y in loader:

#         x = x.to(DEVICE)
#         y = y.to(DEVICE)

#         logits = model(x)

#         loss = criterion(logits, y)

#         total_loss += loss.item()

#     return total_loss / len(loader)


In [ ]:
# #Training the model................
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# vocab = build_vocab(train_df)

# train_part, valid_part = train_test_split(
#     train_df,
#     test_size=0.1,
#     random_state=42,
#     stratify=train_df["answer"]
# )

# train_ds = MCQDataset(train_part, vocab, True)
# valid_ds = MCQDataset(valid_part, vocab, True)

# train_loader = DataLoader(
#     train_ds,
#     batch_size=BATCH_SIZE,
#     shuffle=True
# )

# valid_loader = DataLoader(
#     valid_ds,
#     batch_size=BATCH_SIZE
# )

# model = MCQRanker(len(vocab)).to(DEVICE)

# criterion = nn.BCEWithLogitsLoss()

# optimizer = torch.optim.Adam(
#     model.parameters(),
#     lr=LR
# )

# best_loss = 999

# for epoch in range(EPOCHS):

#     train_loss = train_epoch(
#         model,
#         train_loader,
#         optimizer,
#         criterion
#     )

#     valid_loss = evaluate(
#         model,
#         valid_loader,
#         criterion
#     )

#     print(
#         f"Epoch {epoch+1}/{EPOCHS} "
#         f"Train_loss={train_loss:.4f} "
#         f"Validation_loss={valid_loss:.4f}"
#     )

#     if valid_loss < best_loss:

#         best_loss = valid_loss

#         torch.save(
#             {
#                 "model": model.state_dict(),
#                 "vocab": vocab
#             },
#             "best_model.pt"
#         )

In [ ]:
# #Inference
# checkpoint = torch.load(
#     "best_model.pt",
#     map_location=DEVICE
# )

# model.load_state_dict(checkpoint["model"])
# model.eval()

# test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# test_ds = MCQDataset(
#     test_df,
#     vocab,
#     train_mode=False
# )

# test_loader = DataLoader(
#     test_ds,
#     batch_size=256,
#     shuffle=False
# )

# # Store scores for each question
# scores = {}

# with torch.no_grad():

#     for qids, labels, x in test_loader:

#         x = x.to(DEVICE)

#         logits = model(x)

#         probs = torch.sigmoid(logits).cpu().numpy()

#         for qid, label, p in zip(qids, labels, probs):

#             # Convert ID to string for consistency
#             qid = str(qid)

#             if qid not in scores:
#                 scores[qid] = {}

#             scores[qid][label] = float(p)
# predictions = []

# for qid in test_df["id"]:

#     qid = str(qid)

#     # Safety fallback
#     if qid not in scores:

#         predictions.append("A B C")
#         continue

#     ranked = sorted(
#         scores[qid].items(),
#         key=lambda x: x[1],
#         reverse=True
#     )

#     top3 = [label for label, score in ranked[:3]]

#     # If somehow fewer than 3 labels exist
#     while len(top3) < 3:
#         for lbl in ["A", "B", "C", "D", "E"]:
#             if lbl not in top3:
#                 top3.append(lbl)
#             if len(top3) == 3:
#                 break

#     predictions.append(" ".join(top3))

**Submission**

In [ ]:
# submission = pd.DataFrame(
#     {
#         "ID": test_df["id"],
#         "Prediction": predictions
#     }
# )

# submission.to_csv(
#     "submission.csv",
#     index=False
# )

# print(submission.head())
# print("submission.csv is created successfully")

**Model-2**

In [ ]:
# import pandas as pd
# import numpy as np
# import torch
# from torch.utils.data import Dataset, DataLoader
# from torch.optim import AdamW
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# from sklearn.metrics import accuracy_score, f1_score
# import wandb

In [ ]:
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

## Milestone-1 ##

**Q1**:
Calculate the frequency distribution of the correct answer (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [ ]:
# freq = train["answer"].value_counts().sort_index()
# print(freq)



In [ ]:
# result = freq.max() + freq.min()
# print("Sum =", result)

ANS:Sum = 814

**Q2:**
After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [ ]:
# import string
# vocab = set()

# for text in train["prompt"].astype(str):
#     # lowercase
#     text = text.lower()

#     # remove standard punctuation
#     text = text.translate(str.maketrans('', '', string.punctuation))  #Replace nothing,with nothing,Delete all punctuation characters

#     # split by whitespace
#     vocab.update(text.split())

# print("Vocabulary size:", len(vocab))

ANS:Vocabulary size: 859

**Q3:**
Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [ ]:
# from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# prompt = train.loc[train["id"] == 1, "prompt"].iloc[0]

# # clean
# prompt = prompt.lower()
# prompt = prompt.translate(str.maketrans('', '', string.punctuation))

# # remove stop words
# words = [word for word in prompt.split() if word not in ENGLISH_STOP_WORDS]

# print(words)
# print(len(words))

ANS:13

**Q4:**
Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer? 

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# # combine prompt + options
# texts = (
#     train["prompt"] + " " +
#     train["A"] + " " +
#     train["B"] + " " +
#     train["C"] + " " +
#     train["D"] + " " +
#     train["E"]).tolist()

# vect= TfidfVectorizer(stop_words="english")
# X = vect.fit_transform(texts)
# print("Vocabulary size:", len(vect.vocabulary_))
# print("Feature matrix shape:", X.shape)

ANS:Vocabulary size: 2762

**Q5:**
Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [ ]:
# from sklearn.metrics.pairwise import cosine_similarity
# row = train[train["id"] == 1].iloc[0]

# prompt_text = row["prompt"]
# option_A_text = row["A"]

# v_prompt = vect.transform([prompt_text])
# v_A = vect.transform([option_A_text])
# score = cosine_similarity(v_prompt, v_A)[0][0]

# print(round(score, 5))

ANS:0.27202

**Q6:**
Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [ ]:
# vect.fit(texts)
# options = ["A", "B", "C", "D", "E"]
# correct = 0

# for i, row in train.iterrows():
#     prompt_vec = vect.transform([row["prompt"]])

#     similarities = []

#     for opt in options:
#         opt_vec = vect.transform([row[opt]])
#         sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
#         similarities.append(sim)

#     predicted_index = np.argmax(similarities)  #argmax() finds highest value
#     predicted_answer = options[predicted_index]

#     if predicted_answer == row["answer"]:
#         correct += 1
# accuracy = correct / len(train) * 100
# print("Accuracy:", round(accuracy, 2), "%")  #This tells in what % of questions, the option most similar to the prompt was actually correct

ANS:Accuracy: 13.55 %

**Q7:**
If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  
**ANS:**
Correct answer C is at rank 1,

so,MAP@3=1/1​

        =1

**Q8:**
If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

**ANS:**
Rank of B = 2

MAP@3=1/2

=0.5

**Q9:**
The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [ ]:
# freq = train["answer"].value_counts()
# top3 = freq.index[:3].tolist()

# map3 = train["answer"].apply(
#     lambda x: 1 if x==top3[0] else (0.5 if x==top3[1] else (1/3 if x==top3[2] else 0))
# ).mean()

# print(map3)

ANS:0.42125

**Q10:**
The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [ ]:
# def map3_score(true, preds):  #MAP@3 function
#     if true == preds[0]:
#         return 1.0
#     elif true == preds[1]:
#         return 0.5
#     elif true == preds[2]:
#         return 1/3
#     else:
#         return 0.0

# #Evaluate all rows
# scores = []

# for _,row in train.iterrows():

#     prompt_vec = vect.transform([row["prompt"]])

#     sims = []

#     for opt in options:
#         opt_vec = vect.transform([row[opt]])
#         sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
#         sims.append((opt, sim))

#     # sort by similarity (descending)
#     ranked = sorted(sims, key=lambda x: x[1], reverse=True)

#     top3 = [r[0] for r in ranked[:3]]

#     scores.append(map3_score(row["answer"], top3))

# #Final MAP@3
# final_map3 = np.mean(scores)

# print("TF-IDF Pipeline MAP@3:", round(final_map3, 4))

ANS:TF-IDF Pipeline MAP@3: 0.2962